In [ ]:
import pandas as pd
import plotly
import plotly.express as px
from pathlib import Path
from typing import List


def load_parquet_files(directory: Path, prefix: str) -> pd.DataFrame:
    """
    Load and concatenate Parquet files from a directory matching a filename prefix.

    Args:
        directory (Path): Path to the folder containing Parquet files.
        prefix (str): Common prefix of filenames to match (e.g., 'kmd_hq_nairobi_avo_daily').

    Returns:
        pd.DataFrame: Concatenated and chronologically sorted DataFrame of all matched files.
    """
    files = sorted(directory.glob(f"{prefix}-*.parquet"))
    if not files:
        raise FileNotFoundError(f"No files matching prefix '{prefix}' found in {directory}")
    
    df = pd.concat([pd.read_parquet(file) for file in files])
    df["dtm"] = pd.to_datetime(df["dtm"])
    df = df.sort_values("dtm")
    return df


def plot_time_series(df: pd.DataFrame, columns: List[str], time_col: str = "dtm") -> None:
    """
    Plot interactive time series for selected columns.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        columns (List[str]): List of column names to plot.
        time_col (str): Column name containing datetime values.
    """
    fig = px.line(
        df, x=time_col, y=columns,
        labels={"value": "Concentration (µg/m³)", "variable": "Pollutant", time_col: "Time"},
        title="Time Series of PM1, PM2.5, PM10"
    )
    fig.show()


def plot_diurnal_boxplots(df: pd.DataFrame, columns: List[str], time_col: str = "dtm") -> None:
    """
    Plot diurnal boxplots of selected pollutants.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        columns (List[str]): List of pollutant columns to analyze.
        time_col (str): Column name with datetime information to extract hours from.
    """
    df["hour"] = df[time_col].dt.hour
    df_melted = pd.melt(df, id_vars=["hour"], value_vars=columns,
                        var_name="Pollutant", value_name="Concentration")
    
    fig = px.box(
        df_melted, x="hour", y="Concentration", color="Pollutant",
        title="Diurnal Variability of PM Concentrations",
        labels={"hour": "Hour of Day"}
    )
    fig.show()


def main():
    """
    Main function to load, process, and visualize particulate matter data.
    """
    data_dir = Path("/product_data/data/pay/Kenya/NRB/incoming/avo")  # Change to your local directory
    filename_prefix = "kmd_hq_nairobi_avo_daily"
    pollutant_columns = ["pm1", "pm25_conc", "pm10_conc"]

    df = load_parquet_files(data_dir, filename_prefix)
    plot_time_series(df, pollutant_columns)
    plot_diurnal_boxplots(df, pollutant_columns)


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'plotly'